> 랭체인 공식 문서 Tools: <https://docs.langchain.com/oss/python/langchain/tools>

### 도구 호출 에이전트(Tool Calling Agent)

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
model.invoke([HumanMessage("부산은 지금 몇시야?")])

AIMessage(content='지금은 **2024년 5월 15일 수요일 오후 4시 55분**입니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--1d7355d5-9f22-4586-8ef8-254f1573a856-0', usage_metadata={'input_tokens': 8, 'output_tokens': 272, 'total_tokens': 280, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 244}})

#### ZoneInfo 간단 사용법

In [2]:
from datetime import datetime
from zoneinfo import ZoneInfo # uv add tzdata

# 1. ZoneInfo 객체 생성
seoul_tz = ZoneInfo("Asia/Seoul") # 해당 지역의 시간대 정보 (오프셋, 일광 절약 시간제 등)
newyork_tz = ZoneInfo("America/New_York")
print(seoul_tz)
print(newyork_tz)


Asia/Seoul
America/New_York


In [5]:
type(seoul_tz)

zoneinfo.ZoneInfo

In [10]:
# 2. 현재 시간 가져오기 (시간대 정보 적용)
now_seoul = datetime.now(tz=seoul_tz)
now_newyork = datetime.now(tz=newyork_tz)

print(f"서울 현재 시간: {now_seoul}")
print(f"뉴욕 현재 시간: {now_newyork}")

서울 현재 시간: 2026-01-25 01:43:30.283169+09:00
뉴욕 현재 시간: 2026-01-24 11:43:30.283169-05:00


#### 도구 생성

In [11]:
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    target_timezone = ZoneInfo(timezone)
    now = datetime.now(target_timezone).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [ ]:
from langchain.agents import create_agent

# 도구들을 tools 리스트에 추가
tools = [get_current_time,]

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."
)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "부산은 지금 몇시야?"}]},
)

Asia/Seoul (부산) 현재시각 2026-01-25 01:44:18 


In [ ]:
result

{'messages': [HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}, id='dbb12ee9-55c6-44f1-8f24-95e3f7b6bd13'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'8fd4957b-ccb4-434a-90bb-369c567f32fe': 'CqIEAXLI2nyON3fOGhNOBCMqhwzzQeSzN6tiMfGNjZTBG71rljDj2JJSpIXgMgdFlCHhP/wMuWlRzyxVxP9zqfCy77S81BiIQwH6kHhhXhAL3PbWUpSF+tsvFgJzjg7nJkDie2bVEeTI+/KxIv0l7HyqNIOHN4Fovbxcre8z7wknZrTJwSzjhC/9tgxo+H3SiylY5QG8RbMI/wR4fakr//Hb3lGtbgR+6rrXPMFm4YqwWGfQB510zU0AtUu+Im5XDENv/h94bmgIh2MPxiOEzKZHz5LPaijkLgYYCVN6spVMNXbjUwYiOzV8J4aSx+DtYrZe95EJrOEtA6WiQFaqpl5AzJySv0O6jeffQpzy8hcK+nJ2sXTYlsPoeSPGXQxY4PrsWfKDcbPB3HaJsvFGg4ZAqH3OAN4E54ir+BOOVNwjHuPYTBFDN3DPOIiA9hQJS5EJpvt2VBKKR4PX7XXI9rhGGIOSpWasPzXQGJEXRH7SYp/xi4+pYUMhWMWp3tRtKW7KDhBlBPxDIxsFHX/kYrz1UdR56+iRIUw++WR+UM/9P2w1tOYHHTgJICsrApIIhA3laBNf18bDtAFkNCfSqQqbj//3uXgCRE